In [1]:
import os

if not os.path.exists('F5-TTS-Vietnamese'):
    !git clone https://github.com/nguyenthienhy/F5-TTS-Vietnamese

%cd F5-TTS-Vietnamese

!pip install -e . -q
!pip install soundfile torchaudio -q

print("Cài đặt hoàn tất!")

Cloning into 'F5-TTS-Vietnamese'...
remote: Enumerating objects: 263, done.
remote: Counting objects: 100% (122/122), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 263 (delta 80), reused 72 (delta 72), pack-reused 141 (from 1)
Receiving objects: 100% (263/263), 5.15 MiB | 6.46 MiB/s, done.
Resolving deltas: 100% (106/106), done.
/content/F5-TTS-Vietnamese
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.9/97.9 kB 10.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 98.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━

In [2]:
from huggingface_hub import snapshot_download

model_dir = snapshot_download(
    repo_id="hynt/F5-TTS-Vietnamese-ViVoice",
    local_dir="./F5-TTS-Vietnamese-ViVoice"
)

print(f"Model đã tải về: {model_dir}")
!ls -lh F5-TTS-Vietnamese-ViVoice/

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Model đã tải về: /content/F5-TTS-Vietnamese/F5-TTS-Vietnamese-ViVoice
total 5.1G
-rw-r--r-- 1 root root  12K Mar  9 03:16 config.json
-rw-r--r-- 1 root root 5.1G Mar  9 03:17 model_last.pt
-rw-r--r-- 1 root root 2.3K Mar  9 03:16 README.md


In [3]:
!pip install pydub -q

In [12]:
import gradio as gr
import subprocess
import os
import shutil
import re
import soundfile as sf
import numpy as np
from datetime import datetime
from pydub import AudioSegment, silence

VOCAB_FILE = "/content/F5-TTS-Vietnamese/F5-TTS-Vietnamese-ViVoice/vocab.txt"
CKPT_FILE  = "/content/F5-TTS-Vietnamese/F5-TTS-Vietnamese-ViVoice/model_last.pt"
OUTPUT_DIR = "/content/F5-TTS-Vietnamese/tests/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ────────────────────────────────────────────
# 1. TRIM AUDIO MẪU: chỉ giữ 6 giây đầu,
#    cắt bỏ khoảng lặng đầu/cuối
# ────────────────────────────────────────────
def preprocess_ref_audio(src_path, dst_path, max_sec=6):
    audio = AudioSegment.from_file(src_path)

    # Cắt bỏ khoảng lặng đầu/cuối
    chunks = silence.detect_nonsilent(audio, min_silence_len=200, silence_thresh=-40)
    if chunks:
        start_ms = max(0, chunks[0][0] - 100)
        end_ms   = min(len(audio), chunks[-1][1] + 100)
        audio    = audio[start_ms:end_ms]

    # Giới hạn độ dài tối đa
    audio = audio[:max_sec * 1000]

    # Chuẩn hóa âm lượng
    audio = audio.apply_gain(-audio.max_dBFS - 3)

    audio.export(dst_path, format="wav")
    duration = len(audio) / 1000
    return duration


# ────────────────────────────────────────────
# 2. CHIA VĂN BẢN: theo dấu câu, tối đa 100 ký tự
#    (giảm xuống 100 để tránh lặp)
# ────────────────────────────────────────────
def split_text(text, max_chars=100):
    # Tách theo dấu . , ! ? và xuống dòng
    parts = re.split(r'(?<=[.,!?])\s+|\n+', text.strip())

    chunks = []
    current = ""
    for p in parts:
        p = p.strip()
        if not p:
            continue
        # Nếu 1 câu đã quá dài, tách thêm theo dấu phẩy
        if len(p) > max_chars:
            sub_parts = re.split(r'(?<=,)\s+', p)
            for sp in sub_parts:
                if len(current) + len(sp) + 1 <= max_chars:
                    current += (" " if current else "") + sp
                else:
                    if current:
                        chunks.append(current)
                    current = sp
        else:
            if len(current) + len(p) + 1 <= max_chars:
                current += (" " if current else "") + p
            else:
                if current:
                    chunks.append(current)
                current = p

    if current:
        chunks.append(current)

    # Đảm bảo mỗi chunk kết thúc bằng dấu câu
    result = []
    for c in chunks:
        c = c.strip()
        if c and not c[-1] in '.!?,':
            c += '.'
        if c:
            result.append(c)

    return result if result else [text]


# ────────────────────────────────────────────
# 3. XỬ LÝ OUTPUT: cắt phần lặp ở cuối
#    bằng cách detect silence
# ────────────────────────────────────────────
def trim_output_audio(path):
    try:
        audio = AudioSegment.from_wav(path)
        # Cắt khoảng lặng cuối
        chunks = silence.detect_nonsilent(audio, min_silence_len=500, silence_thresh=-45)
        if chunks:
            end_ms = chunks[-1][1] + 200
            audio  = audio[:end_ms]
            audio.export(path, format="wav")
    except Exception:
        pass


# ────────────────────────────────────────────
# 4. GHÉP AUDIO
# ────────────────────────────────────────────
def merge_audio(audio_files, output_path, silence_ms=250):
    arrays, sr = [], None
    pause = None
    for f in audio_files:
        if not os.path.exists(f):
            continue
        data, sample_rate = sf.read(f)
        if sr is None:
            sr    = sample_rate
            pause = np.zeros(int(sr * silence_ms / 1000))
        # Mono hóa nếu stereo
        if data.ndim > 1:
            data = data.mean(axis=1)
        arrays.append(data)
        arrays.append(pause)
    if not arrays:
        return False
    merged = np.concatenate(arrays[:-1])
    sf.write(output_path, merged, sr)
    return True


# ────────────────────────────────────────────
# 5. CHẠY TTS
# ────────────────────────────────────────────
def run_tts(ref_path, ref_text, gen_text, speed, output_path):
    cmd = [
        "f5-tts_infer-cli",
        "--model",        "F5TTS_Base",
        "--ref_audio",    ref_path,
        "--gen_text",     gen_text,
        "--speed",        str(speed),
        "--vocoder_name", "vocos",
        "--vocab_file",   VOCAB_FILE,
        "--ckpt_file",    CKPT_FILE,
        "--output_file",  output_path,
    ]
    if ref_text.strip():
        cmd += ["--ref_text", ref_text]
    return subprocess.run(cmd, capture_output=True, text=True)


# ────────────────────────────────────────────
# 6. HÀM CHÍNH
# ────────────────────────────────────────────
def clone_voice(ref_audio, ref_text, gen_text, speed, progress=gr.Progress()):
    if ref_audio is None:
        return None, "❌ Vui lòng upload audio tham chiếu!"
    if not gen_text.strip():
        return None, "❌ Vui lòng nhập văn bản muốn tổng hợp!"

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # Preprocess ref audio
    progress(0.05, desc="✂️ Đang cắt chuẩn audio mẫu...")
    ref_path = f"{OUTPUT_DIR}/ref_{timestamp}.wav"
    try:
        duration = preprocess_ref_audio(ref_audio, ref_path, max_sec=6)
    except Exception as e:
        shutil.copy(ref_audio, ref_path)
        duration = -1

    # Chia văn bản
    chunks = split_text(gen_text, max_chars=100)
    total  = len(chunks)
    progress(0.1, desc=f"📝 Chia thành {total} đoạn...")

    chunk_files = []
    for i, chunk in enumerate(chunks):
        chunk_path = f"{OUTPUT_DIR}/chunk_{timestamp}_{i}.wav"
        progress(0.1 + (i + 1) / total * 0.75, desc=f"🧠 Đoạn {i+1}/{total}: '{chunk[:30]}...'" )

        result = run_tts(ref_path, ref_text, chunk, speed, chunk_path)

        if os.path.exists(chunk_path):
            # Cắt phần lặp ở cuối mỗi đoạn
            trim_output_audio(chunk_path)
            chunk_files.append(chunk_path)
        else:
            for f in chunk_files:
                if os.path.exists(f): os.remove(f)
            if os.path.exists(ref_path): os.remove(ref_path)
            err = result.stderr[-400:] if result.stderr else "Không rõ lỗi"
            return None, f"❌ Lỗi đoạn {i+1}:\n{err}"

    # Ghép
    progress(0.9, desc="🔗 Đang ghép audio...")
    final_path = f"{OUTPUT_DIR}/output_{timestamp}.wav"

    if len(chunk_files) == 1:
        shutil.copy(chunk_files[0], final_path)
    else:
        merge_audio(chunk_files, final_path, silence_ms=250)

    # Dọn file tạm
    for f in chunk_files:
        if os.path.exists(f): os.remove(f)
    if os.path.exists(ref_path): os.remove(ref_path)

    progress(1.0, desc="✅ Hoàn tất!")

    if os.path.exists(final_path):
        ref_info = f" | Audio mẫu: {duration:.1f}s" if duration > 0 else ""
        return final_path, f"✅ Thành công! ({total} đoạn ghép lại{ref_info})"
    else:
        return None, "❌ Không tạo được file output."


def reset_form():
    return None, "", "", 1.0, None, ""


# ────────────────────────────────────────────
# GIAO DIỆN
# ────────────────────────────────────────────
with gr.Blocks(title="🎙️ F5-TTS Vietnamese", theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🎙️ F5-TTS Vietnamese — Clone Giọng Nói
    > **Model:** `hynt/F5-TTS-Vietnamese-ViVoice`
    """)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("## 📥 Đầu vào")
            ref_audio = gr.Audio(
                label="🎙️ Audio tham chiếu (tối đa 6 giây, 1 người nói, không nhạc nền)",
                type="filepath",
            )
            ref_text = gr.Textbox(
                label="📝 Transcript của audio tham chiếu (bắt buộc điền để tránh lặp)",
                placeholder="Nhập CHÍNH XÁC nội dung trong audio mẫu...",
                lines=2,
            )
            gr.Markdown("___")
            gen_text = gr.Textbox(
                label="💬 Văn bản muốn tổng hợp",
                placeholder="Nhập văn bản tiếng Việt đầy đủ dấu câu.\nVí dụ: Xin chào, tôi là trợ lý AI.",
                lines=5,
            )
            speed = gr.Slider(
                minimum=0.5, maximum=2.0, value=1.0, step=0.05,
                label="⚡ Tốc độ đọc",
                info="0.5 = chậm  |  1.0 = bình thường  |  2.0 = nhanh",
            )
            with gr.Row():
                btn_clone = gr.Button("🚀 Tạo giọng nói", variant="primary", scale=3)
                btn_reset = gr.Button("🔄 Reset", variant="secondary", scale=1)

        with gr.Column(scale=1):
            gr.Markdown("## 📤 Kết quả")
            out_audio = gr.Audio(
                label="🔊 Audio đã tổng hợp",
                type="filepath",
                show_download_button=True,
            )
            status_box = gr.Textbox(
                label="📊 Trạng thái",
                interactive=False,
                lines=3,
            )
            gr.Markdown("""
            ---
            ### 💡 Checklist để tránh lặp giọng
            - ✅ Audio mẫu **dưới 6 giây**, rõ ràng
            - ✅ **Phải điền transcript** của audio mẫu
            - ✅ Văn bản có **đầy đủ dấu câu** (. , ! ?)
            - ✅ Không để `gen_text` giống nội dung `ref_text`
            - ✅ Nếu vẫn lặp → thử **đổi audio mẫu** khác
            """)

    btn_clone.click(
        fn=clone_voice,
        inputs=[ref_audio, ref_text, gen_text, speed],
        outputs=[out_audio, status_box],
        show_progress=True,
    )
    btn_reset.click(
        fn=reset_form,
        inputs=[],
        outputs=[ref_audio, ref_text, gen_text, speed, out_audio, status_box],
    )

demo.launch(share=True, show_error=True)

/tmp/ipykernel_352/3183195578.py:224: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="🎙️ F5-TTS Vietnamese", theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5f74d83fc50a659f45.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [5]:
import shutil, os

src = "/content/F5-TTS-Vietnamese/F5-TTS-Vietnamese-ViVoice/config.json"
dst = "/content/F5-TTS-Vietnamese/F5-TTS-Vietnamese-ViVoice/vocab.txt"

shutil.copy(src, dst)

print("✅ Đã copy config.json → vocab.txt")
print(f"📄 Kích thước file: {os.path.getsize(dst)} bytes")

# Xác nhận lại
VOCAB_FILE = "/content/F5-TTS-Vietnamese/F5-TTS-Vietnamese-ViVoice/vocab.txt"
CKPT_FILE  = "/content/F5-TTS-Vietnamese/F5-TTS-Vietnamese-ViVoice/model_last.pt"

assert os.path.exists(VOCAB_FILE)
assert os.path.exists(CKPT_FILE)
print("✅ Đủ cả 2 file! Chạy lại Cell Gradio là được.")

✅ Đã copy config.json → vocab.txt
📄 Kích thước file: 11337 bytes
✅ Đủ cả 2 file! Chạy lại Cell Gradio là được.


In [6]:
import os

# Tạo thư mục mà CLI mặc định cần
os.makedirs("/content/F5-TTS-Vietnamese/tests/outputs", exist_ok=True)

# Tạo thư mục output của mình
os.makedirs("/content/F5-TTS-Vietnamese/outputs", exist_ok=True)

print("✅ Đã tạo đủ thư mục!")

✅ Đã tạo đủ thư mục!


In [7]:
# Tạo đúng thư mục mà f5-tts mặc định dùng
OUTPUT_DIR = "/content/F5-TTS-Vietnamese/tests/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [8]:
!which f5-tts_infer-cli
!find / -name "f5-tts_infer-cli" 2>/dev/null

/usr/local/bin/f5-tts_infer-cli
/usr/local/bin/f5-tts_infer-cli


In [9]:
%cd /content/F5-TTS-Vietnamese
!pip install -e . -q
!pip install soundfile torchaudio pydub -q
print("✅ Cài xong!")

/content/F5-TTS-Vietnamese
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for f5-tts (pyproject.toml) ... done
✅ Cài xong!


In [ ]:
import gradio as gr
import subprocess
import os
import shutil
import re
import soundfile as sf
import numpy as np
from datetime import datetime
from pydub import AudioSegment, silence
from concurrent.futures import ThreadPoolExecutor, as_completed

VOCAB_FILE = "/content/F5-TTS-Vietnamese/F5-TTS-Vietnamese-ViVoice/vocab.txt"
CKPT_FILE  = "/content/F5-TTS-Vietnamese/F5-TTS-Vietnamese-ViVoice/model_last.pt"
OUTPUT_DIR = "/content/F5-TTS-Vietnamese/tests/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Số luồng xử lý song song tối đa
MAX_WORKERS = 2


# ────────────────────────────────────────────
# 1. TRIM AUDIO MẪU: chỉ giữ 6 giây đầu,
#    cắt bỏ khoảng lặng đầu/cuối
# ────────────────────────────────────────────
def preprocess_ref_audio(src_path, dst_path, max_sec=6):
    audio = AudioSegment.from_file(src_path)

    # Cắt bỏ khoảng lặng đầu/cuối
    chunks = silence.detect_nonsilent(audio, min_silence_len=200, silence_thresh=-40)
    if chunks:
        start_ms = max(0, chunks[0][0] - 100)
        end_ms   = min(len(audio), chunks[-1][1] + 100)
        audio    = audio[start_ms:end_ms]

    # Giới hạn độ dài tối đa
    audio = audio[:max_sec * 1000]

    # Chuẩn hóa âm lượng
    audio = audio.apply_gain(-audio.max_dBFS - 3)

    audio.export(dst_path, format="wav")
    duration = len(audio) / 1000
    return duration

def split_text(text, max_chars=100):
    # Tách theo dấu . , ! ? và xuống dòng
    parts = re.split(r'(?<=[.,!?])\s+|\n+', text.strip())

    chunks = []
    current = ""
    for p in parts:
        p = p.strip()
        if not p:
            continue
        # Nếu 1 câu đã quá dài, tách thêm theo dấu phẩy
        if len(p) > max_chars:
            sub_parts = re.split(r'(?<=,)\s+', p)
            for sp in sub_parts:
                if len(current) + len(sp) + 1 <= max_chars:
                    current += (" " if current else "") + sp
                else:
                    if current:
                        chunks.append(current)
                    current = sp
        else:
            if len(current) + len(p) + 1 <= max_chars:
                current += (" " if current else "") + p
            else:
                if current:
                    chunks.append(current)
                current = p

    if current:
        chunks.append(current)

    # Đảm bảo mỗi chunk kết thúc bằng dấu câu
    result = []
    for c in chunks:
        c = c.strip()
        if c and not c[-1] in '.!?,':
            c += '.'
        if c:
            result.append(c)

    return result if result else [text]


# ────────────────────────────────────────────
# 3. XỬ LÝ OUTPUT: cắt phần lặp ở cuối
#    bằng cách detect silence
# ────────────────────────────────────────────
def trim_output_audio(path):
    try:
        audio = AudioSegment.from_wav(path)
        # Cắt khoảng lặng cuối
        chunks = silence.detect_nonsilent(audio, min_silence_len=500, silence_thresh=-45)
        if chunks:
            end_ms = chunks[-1][1] + 200
            audio  = audio[:end_ms]
            audio.export(path, format="wav")
    except Exception:
        pass


# ────────────────────────────────────────────
# 4. GHÉP AUDIO
# ────────────────────────────────────────────
def merge_audio(audio_files, output_path, silence_ms=250):
    arrays, sr = [], None
    pause = None
    for f in audio_files:
        if not os.path.exists(f):
            continue
        data, sample_rate = sf.read(f)
        if sr is None:
            sr    = sample_rate
            pause = np.zeros(int(sr * silence_ms / 1000))
        # Mono hóa nếu stereo
        if data.ndim > 1:
            data = data.mean(axis=1)
        arrays.append(data)
        arrays.append(pause)
    if not arrays:
        return False
    merged = np.concatenate(arrays[:-1])
    sf.write(output_path, merged, sr)
    return True


# ────────────────────────────────────────────
# 5. CHẠY TTS
# ────────────────────────────────────────────
def run_tts(ref_path, ref_text, gen_text, speed, output_path):
    cmd = [
        "f5-tts_infer-cli",
        "--model",        "F5TTS_Base",
        "--ref_audio",    ref_path,
        "--gen_text",     gen_text,
        "--speed",        str(speed),
        "--vocoder_name", "vocos",
        "--vocab_file",   VOCAB_FILE,
        "--ckpt_file",    CKPT_FILE,
        "--output_file",  output_path,
    ]
    if ref_text.strip():
        cmd += ["--ref_text", ref_text]
    return subprocess.run(cmd, capture_output=True, text=True)


# ────────────────────────────────────────────
# 5b. XỬ LÝ MỘT ĐOẠN (dùng cho parallel)
# ────────────────────────────────────────────
def process_chunk(args):
    """
    Xử lý một đoạn văn bản và trả về (index, chunk_path, error_msg).
    error_msg là None nếu thành công.
    """
    i, chunk, ref_path, ref_text, speed, chunk_path = args
    result = run_tts(ref_path, ref_text, chunk, speed, chunk_path)
    if os.path.exists(chunk_path):
        trim_output_audio(chunk_path)
        return i, chunk_path, None
    else:
        err = result.stderr[-400:] if result.stderr else "Không rõ lỗi"
        return i, chunk_path, f"❌ Lỗi đoạn {i+1}:\n{err}"


# ────────────────────────────────────────────
# 6. HÀM CHÍNH
# ────────────────────────────────────────────
def clone_voice(ref_audio, ref_text, gen_text, speed, progress=gr.Progress()):
    if ref_audio is None:
        return None, "❌ Vui lòng upload audio tham chiếu!"
    if not gen_text.strip():
        return None, "❌ Vui lòng nhập văn bản muốn tổng hợp!"

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # Preprocess ref audio
    progress(0.05, desc="✂️ Đang cắt chuẩn audio mẫu...")
    ref_path = f"{OUTPUT_DIR}/ref_{timestamp}.wav"
    try:
        duration = preprocess_ref_audio(ref_audio, ref_path, max_sec=6)
    except Exception as e:
        shutil.copy(ref_audio, ref_path)
        duration = -1

    # Chia văn bản
    chunks = split_text(gen_text, max_chars=100)
    total  = len(chunks)
    progress(0.1, desc=f"📝 Chia thành {total} đoạn — bắt đầu xử lý song song ({MAX_WORKERS} luồng)...")

    # Chuẩn bị args cho từng đoạn
    chunk_args = [
        (i, chunk, ref_path, ref_text, speed, f"{OUTPUT_DIR}/chunk_{timestamp}_{i}.wav")
        for i, chunk in enumerate(chunks)
    ]

    # Xử lý song song
    results = {}       # index -> chunk_path
    error_msg = None
    completed = 0

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_idx = {executor.submit(process_chunk, args): args[0] for args in chunk_args}

        for future in as_completed(future_to_idx):
            i, chunk_path, err = future.result()
            completed += 1
            progress(
                0.1 + completed / total * 0.75,
                desc=f"✅ Hoàn thành {completed}/{total} đoạn..."
            )

            if err:
                # Huỷ các future còn lại khi có lỗi
                for f in future_to_idx:
                    f.cancel()
                error_msg = err
                break

            results[i] = chunk_path

    # Nếu có lỗi, dọn dẹp và trả về
    if error_msg:
        for _, p in results.items():
            if os.path.exists(p): os.remove(p)
        if os.path.exists(ref_path): os.remove(ref_path)
        return None, error_msg

    # Sắp xếp lại theo đúng thứ tự index
    chunk_files = [results[i] for i in sorted(results.keys())]

    # Ghép
    progress(0.9, desc="🔗 Đang ghép audio...")
    final_path = f"{OUTPUT_DIR}/output_{timestamp}.wav"

    if len(chunk_files) == 1:
        shutil.copy(chunk_files[0], final_path)
    else:
        merge_audio(chunk_files, final_path, silence_ms=250)

    # Dọn file tạm
    for f in chunk_files:
        if os.path.exists(f): os.remove(f)
    if os.path.exists(ref_path): os.remove(ref_path)

    progress(1.0, desc="✅ Hoàn tất!")

    if os.path.exists(final_path):
        ref_info = f" | Audio mẫu: {duration:.1f}s" if duration > 0 else ""
        return final_path, f"✅ Thành công! ({total} đoạn xử lý song song{ref_info})"
    else:
        return None, "❌ Không tạo được file output."


def reset_form():
    return None, "", "", 1.0, None, ""


# ────────────────────────────────────────────
# GIAO DIỆN
# ────────────────────────────────────────────
with gr.Blocks(title="🎙️ F5-TTS Vietnamese", theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🎙️ F5-TTS Vietnamese — Clone Giọng Nói
    > **Model:** `hynt/F5-TTS-Vietnamese-ViVoice`
    """)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("## 📥 Đầu vào")
            ref_audio = gr.Audio(
                label="🎙️ Audio tham chiếu (tối đa 6 giây, 1 người nói, không nhạc nền)",
                type="filepath",
            )
            ref_text = gr.Textbox(
                label="📝 Transcript của audio tham chiếu (bắt buộc điền để tránh lặp)",
                placeholder="Nhập CHÍNH XÁC nội dung trong audio mẫu...",
                lines=2,
            )
            gr.Markdown("___")
            gen_text = gr.Textbox(
                label="💬 Văn bản muốn tổng hợp",
                placeholder="Nhập văn bản tiếng Việt đầy đủ dấu câu.\nVí dụ: Xin chào, tôi là trợ lý AI.",
                lines=5,
            )
            speed = gr.Slider(
                minimum=0.5, maximum=2.0, value=1.0, step=0.05,
                label="⚡ Tốc độ đọc",
                info="0.5 = chậm  |  1.0 = bình thường  |  2.0 = nhanh",
            )
            with gr.Row():
                btn_clone = gr.Button("🚀 Tạo giọng nói", variant="primary", scale=3)
                btn_reset = gr.Button("🔄 Reset", variant="secondary", scale=1)

        with gr.Column(scale=1):
            gr.Markdown("## 📤 Kết quả")
            out_audio = gr.Audio(
                label="🔊 Audio đã tổng hợp",
                type="filepath",
                show_download_button=True,
            )
            status_box = gr.Textbox(
                label="📊 Trạng thái",
                interactive=False,
                lines=3,
            )
            gr.Markdown("""
            ---
            ### 💡 Checklist để tránh lặp giọng
            - ✅ Audio mẫu **dưới 6 giây**, rõ ràng
            - ✅ **Phải điền transcript** của audio mẫu
            - ✅ Văn bản có **đầy đủ dấu câu** (. , ! ?)
            - ✅ Không để `gen_text` giống nội dung `ref_text`
            - ✅ Nếu vẫn lặp → thử **đổi audio mẫu** khác
            """)

    btn_clone.click(
        fn=clone_voice,
        inputs=[ref_audio, ref_text, gen_text, speed],
        outputs=[out_audio, status_box],
        show_progress=True,
    )
    btn_reset.click(
        fn=reset_form,
        inputs=[],
        outputs=[ref_audio, ref_text, gen_text, speed, out_audio, status_box],
    )

demo.launch(share=True, show_error=True)